Link to the model: https://drive.google.com/file/d/15N5L6AoOLRwKNYTZ0OLwiKphtISqqUe0/view?usp=drive_link

In [4]:
import spacy
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics import classification_report, confusion_matrix
from spacy.tokens import Doc
import joblib

In [5]:
def parse_conll(filepath):
    """
    Parse a CoNLL-formatted file into a list of sentences.
    Args:
        filepath (str): Path to the CoNLL file.

    Returns:
        list[list[list[str]]]: A list of sentences. Each sentence is a list
        of token rows, and each row is a list of column values.
    """
    sentences = []
    current_sentence = []
    
    with open(filepath, 'r', encoding='utf-8') as f: 
        for line in f:
            line = line.strip() #remove whitespace
            
            if line.startswith('#'):
                continue  # skip comment lines
            
            if line == '':  #Empty line shows end of sentence
                if current_sentence:
                    sentences.append(current_sentence)
                    current_sentence = []
                continue
            
            cols = line.split('\t')
            current_sentence.append(cols)
    
    if current_sentence:
        sentences.append(current_sentence)
    
    return sentences

In [6]:
def clean_label(label):
    """
    Maps '_' (no semantic role) and predicates itself to 'O' (outside).
    Args:
        label (str): Original label from the dataset.

    Returns:
        str: Normalized label.
    """
    if label == '_' or label == 'V':
        return 'O'
    return label


def preprocess(sentences):
    """
    Convert parsed CoNLL sentences into SRL instances (one per predicate)
    Args:
        sentences (list[list[list[str]]]): Output from `parse_conll`.

    Returns:
        list[dict]: A list of SRL instances, each containing:
            {'tokens': list[str],      
                'predicate_idx': int,     
                'labels': list[str],       
                'sent': original sentence}  
    """
    instances = []

    for sent in sentences:
        #Find indices of predicates
        predicate_indices = [
            i for i, tok in enumerate(sent)
            if len(tok) > 10 and tok[10] != '_'
        ]

        if not predicate_indices:   # skip sents without predicates
            continue
        # Create one instance per predicate
        for pred_num, pred_idx in enumerate(predicate_indices):
            tokens = [tok[1] for tok in sent] #Extract tokens (column 1)
            labels = []

            for tok in sent:
                column_index = 11 + pred_num  #argument column for this predicate

                # Check if this token even has this column
                if len(tok) <= column_index:
                    label = 'O'

                #if the column value is '_' or 'V', map to 'O'
                elif tok[column_index] == '_' or tok[column_index] == 'V':
                    label = 'O'

                # Otherwise use the actual SRL label
                else:
                    label = tok[column_index]

                labels.append(label)

            instances.append({
                'tokens': tokens,
                'predicate_idx': pred_idx,
                'labels': labels,
                'sent': sent
            })

    return instances

In [7]:
train_sentences = parse_conll('data/en_ewt-up-train.conllu')
test_sentences = parse_conll('data/en_ewt-up-test.conllu')
train_instances = preprocess(train_sentences)
test_instances  = preprocess(test_sentences)

In [8]:
def count_tokens(instances):
    return sum(len(i['tokens']) for i in instances)

def count_sentences(instances):
    return len(instances)

print("train")
print(f"Sentences before preprocessing : {len(train_sentences)}")
print(f"Sentences after preprocessing  : {len(train_instances)}")
print(f"Tokens before preprocessing    : {sum(len(s) for s in train_sentences)}")
print(f"Tokens after preprocessing     : {count_tokens(train_instances)}")

print("\ntest")
print(f"Sentences before preprocessing : {len(test_sentences)}")
print(f"Sentences after preprocessing  : {len(test_instances)}")
print(f"Tokens before preprocessing    : {sum(len(s) for s in test_sentences)}")
print(f"Tokens after preprocessing     : {count_tokens(test_instances)}")

train
Sentences before preprocessing : 12543
Sentences after preprocessing  : 40482
Tokens before preprocessing    : 204609
Tokens after preprocessing     : 1028264

test
Sentences before preprocessing : 2077
Sentences after preprocessing  : 4799
Tokens before preprocessing    : 25097
Tokens after preprocessing     : 101152


In [9]:
nlp = spacy.load("en_core_web_md")

def get_spacy_parse(tokens):
    """
    Parses a list of tokens using spaCy without retokenizing.
    Args:
        tokens (list[str]): A list of pre-tokenized words representing a sentence.

    Returns:
        list[list[str]]: A list of token rows in a CoNLL-style format.
        Each row has 11 columns, and only some are filled:
            row[1] : token text
            row[2] : lemma
            row[3] : POS tag
            row[6] : syntactic head index (1-based)
            row[7] : dependency relation label
        Other columns are empty placeholders.
    """
    #create Doc directly from tokens(spaCy skips its tokenizer)
    doc = Doc(nlp.vocab, words=tokens)
    # Run the spaCy pipeline
    doc = nlp(doc)
    
    parsed_sent = []
    for token in doc:
        # Create an empty row with 11 columns to match CoNLL structure
        row = [''] * 11
        row[1] = token.text  #word form
        row[2] = token.lemma_ #lemma
        row[3] = token.pos_  #POS tag
        row[6] = str(token.head.i + 1)  # # head index (1-indexed to match CoNLL)
        row[7] = token.dep_  #dep rel label
        parsed_sent.append(row)
    
    return parsed_sent

## Feature Extraction
Three features are extracted per token for each instance. All features are computed using spaCy on raw token list, to make feature extraction self-sufficient.

### Feature 1: Directed Dependency Path to Predicate + Predicate Lemma
The syntactic path between a token and its predicate directly encodes their syntactic relationship, which can strongly correalate with semantic roles. For example, a token with 'nsubj↑' is almost always ARG0, and 'obj↑' is almost always ARG1. The directionality (↑ up, ↓ down) shows whether the token is a dependent or head relative to the predicate. Appending the predicate lemma allows the model to learn patterns specific to the predicate (e.g. 'nsubj↑|kill' vs 'nsubj↑|give'). For Logistic Regression, this is
represented as a string and one-hot encoded, ao it provides an independent binary feature with a learnable weight.

### Feature 2: Token Lemma
Certain lemmas are strongly associated with specific roles: pronouns like 'he' are often ARG0, nominals like 'damage' are often ARG1s, and temporal nouns like 'year' are often ARGM-TMP. By using the lemma rather than surface forms, we can reduce sparsity. This feature complements the path feature. Two tokens can have the same dependency path but different lemmas and therefore different roles. This is a categorical string, one-hot encoded for Logistic Regression.

### Feature 3: Token POS tag
The universal POS tag of the token strongly correlates with semantic roles: ARG0 is almost always a NOUN or PRON, ARGM-ADV is almost always an ADV, and ARGM-MOD is almost always an AUX. Punctuation tokens (PUNCT) are almost never arguments (outside label). POS complements the dependency path features. It describes what POS the token is, while the path describes how it relates to the predicate. This is a categorical feature that is one-hot encoded, suitable for Logistic Regression. There are 17 universal POS tags so it is also not sparse. 

In [10]:
def get_dep_path_to_predicate(sent, token_idx, predicate_idx):
    """
    Get the dependency path from a token to a predicate, 
    by going upward from the token to the Lowest Common Ancestor (LCA), and downward from the LCA to the predicate.

    Directions are encoded as:
        ↑ : moving up the dependency tree (toward the root)
        ↓ : moving down the dependency tree (toward the predicate)
        
    The predicate lemma is appended to the path to use predicate information.
    
    Args:
        sent (list[list[str]]): A sentence in CoNLL-like format.
        token_idx (int): Index of the token.
        predicate_idx (int): Index of the predicate token.

    Returns: str: Dependency path in the form of "dep↑>dep↑>dep↓>dep↓|predicate_lemma"
    """
    # Get the predicate lemma
    predicate_lemma = sent[predicate_idx][2]
     # If the token itself is the predicate
    if token_idx == predicate_idx:
        return f"SELF|{predicate_lemma}"
    
    head   = {}
    deprel = {}
    # Build head and dependency relation dictionaries
    for i, tok in enumerate(sent):
        # safely parse head index 
        try:
            head[i] = int(tok[6]) - 1
        except (ValueError, IndexError):
            head[i] = -1  # treat missing value as root
        deprel[i] = tok[7] if len(tok) > 7 else 'UNK' # Get dependency relation
    
    def get_ancestors(idx):
        """
        Go upward in the dependency tree and collect ancestors.
        Returns a list of (node_index, dependency_relation) pairs.
        """
        path = []
        visited = set()
        current = idx
        while current != -1:
            if current in visited:
                break
            visited.add(current)
            path.append((current, deprel.get(current, '')))
            current = head.get(current, -1)
        return path
    # Get ancestor paths for token and predicate
    token_ancestors = get_ancestors(token_idx)
    pred_ancestors = get_ancestors(predicate_idx)
    # Set of predicate ancestor nodes
    pred_ancestor_nodes = {node for node, dep in pred_ancestors}
    # Find Lowest Common Ancestor (LCA)
    lca = None
    token_path_up = []
    for node, dep in token_ancestors:
        if node in pred_ancestor_nodes:
            lca = node
            break
        token_path_up.append(dep + "↑")
    # If no common ancestor exists
    if lca is None:
        return f"NOPATH|{predicate_lemma}"
    # Build path from predicate up to LCA
    pred_path_up = []
    for node, dep in pred_ancestors:
        if node == lca:
            break
        pred_path_up.append(dep + "↓")
    pred_path_down = list(reversed(pred_path_up)) #Reverse to get LCA → predicate direction
    # Combine upward and downward paths
    full_path = token_path_up + pred_path_down
    # Convert to string 
    path_str = ">".join(full_path) if full_path else "SELF"
    
    return f"{path_str}|{predicate_lemma}"

In [11]:
def extract_features(instance):
    """
    Extract features for a SRL instance, including:
        - dependency path from the token to the predicate+predicate's lemma
        - token lemma
        - token POS tag
    Args:
        instance (dict): An instance produced by `preprocess`
    Returns: list[dict]
    """
    tokens = instance['tokens']
    pred_idx = instance['predicate_idx']
    # Parse the sentence with spaCy
    parsed_sent = get_spacy_parse(tokens)  
    # Extract features for each token
    features = []
    for i in range(len(tokens)):
        feat = {
            'dep_path':   get_dep_path_to_predicate(parsed_sent, i, pred_idx),
            'token_lemma': parsed_sent[i][2],   #lemma from spaCy
            'token_pos':   parsed_sent[i][3],   #POS from spaCy
        }
        features.append(feat)
    
    return features

In [12]:
# extract features and labels 
train_features = []
train_labels = []

for instance in train_instances:
    # Extract feature dictionaries for each token
    train_features.extend(extract_features(instance))
    # Append the gold labels
    train_labels.extend(instance['labels'])

test_features = []
test_labels = []

for instance in test_instances:
    # Extract token-level features
    test_features.extend(extract_features(instance))
    # Append gold labels (only used for evaluation, not training)
    test_labels.extend(instance['labels'])

# vectorize
vec = DictVectorizer()
X_train = vec.fit_transform(train_features)   #fit on train
X_test = vec.transform(test_features)        # transform on test
#labels (list)
y_train = train_labels
y_test = test_labels
#stats
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape:  {X_test.shape}")
print(f"Number of unique labels: {len(set(y_train))}")

X_train shape: (1028264, 552678)
X_test shape:  (101152, 552678)
Number of unique labels: 59


In [14]:
print("Example features (pre-vectorized)")
for instance in train_instances[:2]:
    feats    = extract_features(instance)
    tokens   = instance['tokens']
    pred_idx = instance['predicate_idx']
    
    print(f"Predicate: '{tokens[pred_idx]}'")
    print(f"{'Token':<15} {'dep_path':<40} {'token_lemma':<15} {'token_pos'}")
    print("-" * 90)
    for tok, feat in zip(tokens, feats):
        print(f"{tok:<15} {feat['dep_path']:<40} {feat['token_lemma']:<15} {feat['token_pos']}")
    print()

# show one vectorized row to confirm these dicts are what the model receives
print("Same features after vectorization (one token row, non-zero columns only):")
example_vec = vec.transform([extract_features(train_instances[0])[0]])
feature_names = vec.get_feature_names_out()
nonzero_idx = example_vec.nonzero()[1]
for idx in nonzero_idx:
    print(f"  {feature_names[idx]} = 1")

Example features (pre-vectorized)
Predicate: 'killed'
Token           dep_path                                 token_lemma     token_pos
------------------------------------------------------------------------------------------
Al              compound↑>dep↑|kill                      Al              PROPN
-               punct↑>dep↑|kill                         -               PUNCT
Zaman           dep↑|kill                                Zaman           PROPN
:               punct↑>dep↑|kill                         :               PUNCT
American        amod↑>nsubj↑|kill                        american        ADJ
forces          nsubj↑|kill                              force           NOUN
killed          SELF|kill                                kill            VERB
Shaikh          compound↑>dobj↑|kill                     Shaikh          PROPN
Abdullah        compound↑>dobj↑|kill                     Abdullah        PROPN
al              compound↑>dobj↑|kill                     al      

In [12]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
print("Training done.")

Training done.


In [14]:
y_pred = model.predict(X_test)

print("Classification Report")
print(classification_report(y_test, y_pred))

# labeled confusion matrix
labels_sorted = sorted(set(y_test))
cm = confusion_matrix(y_test, y_pred, labels=labels_sorted)
cm_df = pd.DataFrame(cm, index=labels_sorted, columns=labels_sorted)

print("Confusion Matrix")
print(cm_df)

Classification Report


C:\Users\Rey\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\Rey\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\Rey\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


              precision    recall  f1-score   support

        ARG0       0.86      0.59      0.70      1733
        ARG1       0.83      0.63      0.72      3241
    ARG1-DSP       0.00      0.00      0.00         4
        ARG2       0.78      0.62      0.69      1129
        ARG3       0.77      0.23      0.35        74
        ARG4       0.44      0.48      0.46        56
        ARG5       0.00      0.00      0.00         1
        ARGA       0.00      0.00      0.00         2
    ARGM-ADJ       0.78      0.41      0.54       228
    ARGM-ADV       0.61      0.23      0.34       496
    ARGM-CAU       0.50      0.13      0.21        46
    ARGM-COM       0.00      0.00      0.00        13
    ARGM-CXN       1.00      0.08      0.15        12
    ARGM-DIR       0.40      0.09      0.14        47
    ARGM-DIS       0.79      0.30      0.44       182
    ARGM-EXT       0.86      0.42      0.56       105
    ARGM-GOL       0.00      0.00      0.00        24
    ARGM-LOC       0.66    

## Results Discussion
The model has 95% overall accuracy, but this is influenced by the O label (no role), which largely dominates the dataset. The weighted average F1 of 0.94 also reflects this class imbalance, and is not a good measure of SRL performance on argument roles. The core argument roles ARG0 (F1=0.70) and ARG1 (F1=0.72) are the strongest performing argument labels. This is expected because they are the most frequent roles and have string correlations with syntax (nsubj for ARG0, obj for ARG1) that the dependency path feature captures. ARGM-MOD (F1=0.83) and ARGM-NEG (F1=0.76) also perform well, likely because modal auxiliaries and negation markers have distinct POS tags and dependency relations.

Adjunct roles have low performance. ARGM-ADV (F1=0.34), ARGM-LOC (F1=0.22), ARGM-MNR (F1=0.15) and ARGM-PRP (F1=0.13). These roles are syntactically diverse (a location argument can be a prepositional phrase, an adverb, or a clause) which makes it hard to generalize from dependency paths. The confusion matrix also shows that ARGM-ADV, ARGM-LOC, and ARGM-TMP frequently get predicted as O (majority class).

Rare roles with few examples like ARG5 (1 instance), ARGA (2), C-ARG0 (3), and R-ARGMs score F1=0. The model never sees enough training examples to learn these patterns. The macro average F1 of 0.25 also shows this. It treats all labels equally and is lowered by the zero scores of rare labels.

In [15]:
# flatten tokens from test instances to match y_pred
all_tokens = []
for instance in test_instances:
    all_tokens.extend(instance['tokens'])

with open('predictions.tsv', 'w', encoding='utf-8') as f:
    f.write("token\tgold\tpredicted\n")
    for token, gold, pred in zip(all_tokens, y_test, y_pred):
        f.write(f"{token}\t{gold}\t{pred}\n")

print("Predictions saved to predictions.tsv")

Predictions saved to predictions.tsv


In [16]:
def predict_srl(sentence_tokens, predicate_mask, model, vectorizer):
    """
    Performs SRL on a new sentence given the predicate position.
    
    Args:
        sentence_tokens: list of strings e.g. ['Pia', 'asked', 'Luis', '.']
        predicate_mask:  list of 0/1    e.g. [0, 1, 0, 0]  (1 = predicate)
        model:           trained LogisticRegression
        vectorizer:      fitted DictVectorizer
    
    Returns:
        list of predicted SRL labels.
    """
    #Get predicate index from the mask
    predicate_idx = predicate_mask.index(1)
    # Create an instance in the same format as training 
    instance = {
        'tokens':        sentence_tokens,
        'predicate_idx': predicate_idx,
    }
    # Extract feature dictionaries using spaCy
    features = extract_features(instance)  
    X = vectorizer.transform(features)
    preds = model.predict(X)
    return list(preds)


#Example
sentence = ['Pia', 'asked', 'Luis', 'to', 'write', 'this', 'sentence', '.']
#"asked" as predicate
print("Predicate: 'asked'")
labels = predict_srl(sentence, [0,1,0,0,0,0,0,0], model, vec)
for tok, lab in zip(sentence, labels):
    print(f"  {tok:<12} {lab}")
#"write" as predicate
print("\nPredicate: 'write'")
labels = predict_srl(sentence, [0,0,0,0,1,0,0,0], model, vec)
for tok, lab in zip(sentence, labels):
    print(f"  {tok:<12} {lab}")

Predicate: 'asked'
  Pia          ARG0
  asked        O
  Luis         ARG2
  to           O
  write        ARG1
  this         O
  sentence     O
  .            O

Predicate: 'write'
  Pia          O
  asked        O
  Luis         O
  to           O
  write        O
  this         O
  sentence     ARG1
  .            O


In [19]:
joblib.dump({'model': model, 'vectorizer': vec}, 'srl_model.joblib')
print("Model saved.")

Model saved.
